In [ ]:
from __future__ import annotations

import fsspec
import pickle

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

import manifold_dynamics.neural_utils as nu
import manifold_dynamics.paths as pth
import manifold_dynamics.spike_response_stats as srs
import manifold_dynamics.tuning_utils as tut
import visionlab_utils.storage as vst

In [ ]:
# ROI name, should be 3 parts (index.label.category)
target = "07.MF1.F"
target_parts = target.split(".")
roi_label = f"{int(target_parts[0]):02d}.{target_parts[1]}.{target_parts[2]}"

# load in multi-session ROI data, binned to PSTH
raster_4d = nu.significant_trial_raster(target, alpha=0.05, bin_size_ms=20)
raster_3d = np.nanmean(raster_4d, axis=3)

# get top-k value for ROI
topk_local = vst.fetch(f"{pth.OTHERS}/topk_vals.pkl")
with open(topk_local, "rb") as f:
    topk_vals = pickle.load(f)
top_k = int(topk_vals[roi_label]["k"])

In [ ]:
ONSET = 50
RESP = slice(ONSET + 50, ONSET + 220)
BASE = slice(ONSET - 50, ONSET + 0)

X = np.array(raster_3d)
resp = np.nanmean(X[:, RESP, :], axis=(0, 1))       # (images,)
base_mean = np.nanmean(X[:, BASE, :], axis=(0, 1))
base_std = np.nanstd(X[:, BASE, :], axis=(0, 1))

# avoid divide-by-zero
base_std = np.where(base_std == 0, np.nan, base_std)

zscores = (resp - base_mean) / base_std
scores = tut.rank_images_by_response(raster_3d)

In [ ]:
avg_resp = np.nanmean(zscores[scores][:top_k])
print(avg_resp)
sns.histplot(zscores)